# Singular Value Decomposition (SVD) and Low-Rank Matrix Factorization

## What is SVD?
Singular Value Decomposition is a fundamental matrix factorization technique that decomposes any matrix into three components:

```
W = U × Σ × V^T
```

Where:
- **U**: Left singular vectors (orthogonal matrix, d×d)
- **Σ**: Diagonal matrix of singular values (d×k), sorted from largest to smallest
- **V^T**: Right singular vectors transposed (orthogonal matrix, k×k)

## Why is SVD Important for LoRA?
SVD helps us understand **low-rank approximations**:
- Many matrices can be well-approximated by lower-rank versions
- We can keep only the top-r singular values to create a rank-r approximation
- This is the mathematical foundation for LoRA's efficiency!

### Visual Representation:
```
Full Matrix W (d×k):          Low-Rank Approximation:
┌─────────────┐              ┌────┐   ┌─────────────┐
│             │              │    │ × │      A      │
│      W      │    ≈         │ B  │   └─────────────┘
│             │              │    │   
│             │              └────┘   
└─────────────┘               d×r          r×k
  d×k params              (d×r + r×k) params!
```

In [2]:
import torch
import numpy as np

# Set random seed for reproducibility
# This ensures we get the same "random" numbers every time
_ = torch.manual_seed(0)

## Step 1: Generate a Rank-Deficient Matrix

### What is Matrix Rank?
The rank of a matrix is the dimension of the vector space spanned by its columns (or rows).
- A matrix is **full rank** if rank = min(rows, columns)
- A matrix is **rank-deficient** if rank < min(rows, columns)

### Why Create a Rank-Deficient Matrix?
Many real-world matrices (like neural network weight updates) are naturally low-rank.
This demonstrates that we can represent them more efficiently!

### The Trick:
```
If we multiply: (d×r) × (r×k) = (d×k)
The result can have at most rank r!
```

We're creating a 10×10 matrix that secretly has only rank 2.

In [3]:
# Define matrix dimensions
d, k = 10, 10  # We'll create a square 10×10 matrix

# This is the KEY TRICK to create a rank-deficient matrix:
# We multiply two smaller matrices to guarantee low rank
W_rank = 2  # Target rank: this matrix will have rank 2

# Generate W by multiplying (10×2) × (2×10) = (10×10)
# Mathematical guarantee: rank(A×B) ≤ min(rank(A), rank(B))
# Since both random matrices have rank 2, their product has rank ≤ 2
W = torch.randn(d, W_rank) @ torch.randn(W_rank, k)

print(W)

tensor([[-1.0797,  0.5545,  0.8058, -0.7140, -0.1518,  1.0773,  2.3690,  0.8486,
         -1.1825, -3.2632],
        [-0.3303,  0.2283,  0.4145, -0.1924, -0.0215,  0.3276,  0.7926,  0.2233,
         -0.3422, -0.9614],
        [-0.5256,  0.9864,  2.4447, -0.0290,  0.2305,  0.5000,  1.9831, -0.0311,
         -0.3369, -1.1376],
        [ 0.7900, -1.1336, -2.6746,  0.1988, -0.1982, -0.7634, -2.5763, -0.1696,
          0.6227,  1.9294],
        [ 0.1258,  0.1458,  0.5090,  0.1768,  0.1071, -0.1327, -0.0323, -0.2294,
          0.2079,  0.5128],
        [ 0.7697,  0.0050,  0.5725,  0.6870,  0.2783, -0.7818, -1.2253, -0.8533,
          0.9765,  2.5786],
        [ 1.4157, -0.7814, -1.2121,  0.9120,  0.1760, -1.4108, -3.1692, -1.0791,
          1.5325,  4.2447],
        [-0.0119,  0.6050,  1.7245,  0.2584,  0.2528, -0.0086,  0.7198, -0.3620,
          0.1865,  0.3410],
        [ 1.0485, -0.6394, -1.0715,  0.6485,  0.1046, -1.0427, -2.4174, -0.7615,
          1.1147,  3.1054],
        [ 0.9088,  

## Step 2: Verify the Matrix Rank

Let's confirm that our matrix really has rank 2, not the maximum possible rank of 10.

### How is Rank Computed?
NumPy's `matrix_rank` function:
1. Performs SVD to get singular values
2. Counts how many singular values are above a small threshold (numerical tolerance)
3. This count is the rank

### What to Expect:
- Full rank matrix: rank = 10
- Our rank-deficient matrix: rank = 2 ✓

In [4]:
# Calculate the rank of W using numpy's linear algebra functions
# This counts the number of linearly independent rows (or columns)
W_rank = np.linalg.matrix_rank(W)
print(f'Rank of W: {W_rank}')

# Success! Even though W is 10×10 (could be rank 10),
# it only has rank 2 because we constructed it that way

Rank of W: 2


## Step 3: Perform SVD Decomposition

Now for the main event: decomposing W using SVD!

### The SVD Formula:
```
W = U × Σ × V^T
```

### For Low-Rank Approximation:
We only keep the top-r singular values and corresponding vectors:
```
W ≈ U_r × Σ_r × V_r^T
```

### Converting to LoRA-style B×A Form:
```
B = U_r × Σ_r  (d×r)
A = V_r^T      (r×k)

Therefore: W ≈ B × A
```

### Visual Breakdown:
```
     W (10×10)    =    U (10×10)  ×  Σ (10×10)  ×  V^T (10×10)
                       ↓ keep r=2      ↓ keep r=2     ↓ keep r=2
     W (10×10)    ≈    U_r (10×2)  ×  Σ_r (2×2)  ×  V_r^T (2×10)
                       └────────────┬─────────────┘
                              B (10×2)           A (2×10)
```

In [6]:
# Perform Singular Value Decomposition on W
# torch.svd returns: U, S (singular values as vector), V
# Mathematical form: W = U @ diag(S) @ V^T
U, S, V = torch.svd(W)

# For a rank-r approximation, we only need:
# - First r columns of U
# - First r singular values
# - First r columns of V

# Extract first r=2 columns of U (the most important left singular vectors)
U_r = U[:, :W_rank]  # Shape: (10, 2)

# Create diagonal matrix with first r=2 singular values
# These values represent the "importance" of each component
S_r = torch.diag(S[:W_rank])  # Shape: (2, 2)

# Extract and transpose first r=2 columns of V to get V_r^T
# .t() transposes the matrix
V_r = V[:, :W_rank].t()  # Shape: (2, 10)

# Combine U_r and S_r into matrix B
# This absorbs the scaling factors (singular values) into B
B = U_r @ S_r  # (10, 2) @ (2, 2) = (10, 2)

# A is simply V_r^T
A = V_r  # Shape: (2, 10)

print(f'Shape of B: {B.shape}')  # (10, 2) - similar to LoRA's B matrix
print(f'Shape of A: {A.shape}')  # (2, 10) - similar to LoRA's A matrix

# Now we have: W ≈ B @ A
# This is exactly the form LoRA uses!

Shape of B: torch.Size([10, 2])
Shape of A: torch.Size([2, 10])


## Step 4: Verify the Decomposition

Let's prove that B×A actually reconstructs W by testing with a real input!

### The Test:
```
Original computation:    y = W × x + bias
Using decomposition:     y' = (B × A) × x + bias
```

If our decomposition is correct, y and y' should be **identical** (or extremely close).

### Why Does This Matter?
This demonstrates that:
1. We can replace a large matrix (W) with smaller matrices (B, A)
2. The output remains the same
3. We use fewer parameters!

This is the core principle behind LoRA's efficiency.

In [10]:
# Generate random bias and input vectors for testing
bias = torch.randn(d)  # Shape: (10,)
x = torch.randn(d)     # Shape: (10,)

# Method 1: Compute output using the original matrix W
# This is the "ground truth" we want to match
y = W @ x + bias

# Method 2: Compute output using the factorized form B×A
# Matrix multiplication is associative: (B×A)×x = B×(A×x)
y_prime = (B @ A) @ x + bias

print("Original y using W:\n", y)
print("")
print("y'computed using BA:\n", y_prime)
print("")

# Check how close they are
max_diff = torch.max(torch.abs(y - y_prime)).item()
print(f"SUCCESS! The outputs are virtually identical!")
print(f"Maximum difference: {max_diff:.2e} (essentially zero due to floating-point precision)")

Original y using W:
 tensor([-4.2808, -1.2647,  2.9707, -0.9784,  0.9193,  3.5211,  5.3049,  5.5870,
         1.8145,  4.7644])

y'computed using BA:
 tensor([-4.2808, -1.2647,  2.9707, -0.9784,  0.9193,  3.5211,  5.3049,  5.5870,
         1.8145,  4.7644])

SUCCESS! The outputs are virtually identical!
Maximum difference: 2.38e-06 (essentially zero due to floating-point precision)


## Step 5: Compare Parameter Counts

Now let's quantify the efficiency gains from this decomposition.

### Parameter Count Formulas:
```
Original matrix W:           d × k parameters
Factorized form (B, A):      (d × r) + (r × k) parameters
```

### When is Factorization Beneficial?
```
We save parameters when:
    (d × r) + (r × k) < d × k
    r × (d + k) < d × k
    r < (d × k) / (d + k)
```

For our case (d=10, k=10, r=2):
```
Original:     10 × 10 = 100 parameters
Factorized:   (10 × 2) + (2 × 10) = 20 + 20 = 40 parameters
Savings:      60 parameters (60% reduction!)
```

### Real-World Impact:
For large neural networks:
- If d=4096, k=4096, r=8:
  - Original: 16,777,216 parameters
  - Factorized: 65,536 parameters
  - **99.6% reduction!** 🚀

In [12]:
# Count total parameters in the original matrix W
params_W = W.nelement()  # nelement() returns total number of elements

# Count total parameters in the factorized form (B + A)
params_BA = B.nelement() + A.nelement()

# Calculate savings
params_saved = params_W - params_BA
reduction_percent = (params_saved / params_W) * 100
compression_ratio = params_W / params_BA

print("═" * 47)
print("           PARAMETER COMPARISON")
print("═" * 47)
print("")
print("Original matrix W:")
print(f"  Shape: {d} × {k}")
print(f"  Total parameters: {params_W}")
print("")
print("Factorized matrices B and A:")
print(f"  B shape: {B.shape[0]} × {B.shape[1]}  →  {B.nelement()} parameters")
print(f"  A shape: {A.shape[0]} × {A.shape[1]}  →  {A.nelement()} parameters")
print(f"  Total parameters: {params_BA}")
print("")
print("═" * 47)
print("EFFICIENCY GAIN:")
print(f"  Parameters saved: {params_saved} ({reduction_percent:.1f}% reduction)")
print(f"  Compression ratio: {compression_ratio:.1f}× fewer parameters")
print("═" * 47)
print("")
print("Key Insight:")
print(f"By using rank-{W_rank} decomposition, we reduced parameters by {reduction_percent:.0f}%")
print("while maintaining identical outputs!")
print("")
print("This is why LoRA is so powerful for fine-tuning large models.")

═══════════════════════════════════════════════
           PARAMETER COMPARISON
═══════════════════════════════════════════════

Original matrix W:
  Shape: 10 × 10
  Total parameters: 100

Factorized matrices B and A:
  B shape: 10 × 2  →  20 parameters
  A shape: 2 × 10  →  20 parameters
  Total parameters: 40

═══════════════════════════════════════════════
EFFICIENCY GAIN:
  Parameters saved: 60 (60.0% reduction)
  Compression ratio: 2.5× fewer parameters
═══════════════════════════════════════════════

Key Insight:
By using rank-2 decomposition, we reduced parameters by 60%
while maintaining identical outputs!

This is why LoRA is so powerful for fine-tuning large models.


## Summary: Connecting SVD to LoRA

### What We Learned:

1. **SVD provides a mathematical foundation** for low-rank approximations
   - Any matrix can be decomposed into U × Σ × V^T
   - We can approximate it using only the top-r singular values

2. **Low-rank factorization dramatically reduces parameters**
   - W (d×k) → B (d×r) and A (r×k)
   - From d×k to r×(d+k) parameters
   - Typical savings: 90-99% for neural networks!

3. **The outputs remain (nearly) identical**
   - W×x ≈ (B×A)×x
   - Perfect reconstruction for truly rank-r matrices
   - Good approximation for approximately low-rank matrices

### How This Relates to LoRA:

```
LoRA's Approach:
┌─────────────────────────────────────────────────┐
│  Original weights: W (frozen)                   │
│  LoRA adaptation:  ΔW = B × A (trainable)       │
│  Final weights:    W' = W + B × A               │
└─────────────────────────────────────────────────┘
```

LoRA assumes that weight **updates** (not the weights themselves) are low-rank.
This notebook demonstrated that low-rank approximations can be very effective!

### Practical Implications:

-  **Memory efficient**: Store only B and A instead of full weight updates
-  **Computation efficient**: Fewer parameters to update during training
-  **Modular**: Can swap different B×A adapters for different tasks
-  **Preserves base model**: Original weights W stay frozen